# Error analysis

Where does the keyword spotter fail, on Speech Commands and on the Serbian-accented set?

Inputs: `runs/<run>/predictions.npz` and `accented_metrics.json` from `scripts/evaluate_accented.py`, for every seed of one experiment.

**Privacy:** the accented recordings are private (see `data_accented/README.md`). This notebook never embeds audio in its outputs, because notebooks are committed. To listen to a clip, call `listen(path)` in a scratch cell and clear its output before saving.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from IPython.display import Audio

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXPERIMENT = "hp_lr3.0e-3"  # the final model (lr 3e-3, 3 seeds)

runs = []
for path in sorted((REPO / "runs").glob("*/predictions.npz")):
    config = yaml.safe_load((path.parent / "config.yaml").read_text())
    if config["name"] == EXPERIMENT:
        runs.append({
            "seed": config["seed"],
            **dict(np.load(path)),
            "metrics": json.loads((path.parent / "accented_metrics.json").read_text()),
        })
LABELS = runs[0]["metrics"]["labels"]
print(f"{EXPERIMENT}: seeds {[r['seed'] for r in runs]}")


def listen(path: str) -> Audio:
    """Play a clip locally. Clear the output before saving: never commit accented audio."""
    root = REPO / "data_accented" if path.startswith("clips/") else REPO / "data" / "speech_commands_v0.02"
    return Audio(filename=str(root / path))

hp_lr3.0e-3: seeds [0, 1, 2]


## 1. Top confused pairs

Counts pooled over seeds. Official: all 12 classes; accented: the 10 target words (there are no silence/unknown clips).

In [2]:
def confusions(which: str) -> pd.DataFrame:
    counts = {}
    for r in runs:
        for t, p in zip(r[f"{which}_y_true"], r[f"{which}_y_pred"]):
            if t != p:
                counts[(LABELS[t], LABELS[p])] = counts.get((LABELS[t], LABELS[p]), 0) + 1
    n = len(runs[0][f"{which}_y_true"]) * len(runs)
    df = pd.DataFrame([(t, p, c) for (t, p), c in counts.items()], columns=["true", "pred", "count"])
    df["share_of_errors"] = df["count"] / df["count"].sum()
    df["per_1000_clips"] = 1000 * df["count"] / n
    return df.sort_values("count", ascending=False).reset_index(drop=True)


official_conf = confusions("official")
accented_conf = confusions("accented")
print("official test:", official_conf["count"].sum() // len(runs), "errors per seed")
display(official_conf.head(10).round(3))
print("accented:", accented_conf["count"].sum() / len(runs), "errors per seed")
display(accented_conf.head(10).round(3))

official test: 190 errors per seed


,true,pred,count,share_of_errors,per_1000_clips
0,on,off,26,0.046,1.773
1,_unknown_,left,24,0.042,1.637
2,right,_unknown_,22,0.039,1.500
3,_unknown_,go,22,0.039,1.500
4,go,no,22,0.039,1.500
5,on,_unknown_,20,0.035,1.364
6,_unknown_,on,20,0.035,1.364
7,down,go,19,0.033,1.296
8,go,down,18,0.032,1.227
9,_unknown_,right,17,0.030,1.159


accented: 28.0 errors per seed


,true,pred,count,share_of_errors,per_1000_clips
0,on,_unknown_,16,0.190,10.458
1,right,_unknown_,15,0.179,9.804
2,down,go,9,0.107,5.882
3,no,_unknown_,7,0.083,4.575
4,left,_unknown_,7,0.083,4.575
5,no,down,4,0.048,2.614
6,go,off,3,0.036,1.961
7,up,_unknown_,3,0.036,1.961
8,off,_unknown_,3,0.036,1.961
9,on,down,2,0.024,1.307


In [3]:
# How much of each set's error is "rejected as _unknown_" vs. "confused with another word"?
def error_kinds(df: pd.DataFrame) -> pd.Series:
    words = df[df["true"].isin(LABELS[:10])]
    kinds = np.where(words["pred"] == "_unknown_", "-> _unknown_",
                     np.where(words["pred"] == "_silence_", "-> _silence_", "-> other word"))
    return words.groupby(kinds)["count"].sum() / words["count"].sum()

pd.DataFrame({"official (target words)": error_kinds(official_conf),
              "accented": error_kinds(accented_conf)}).round(3)

,official (target words),accented
-> _unknown_,0.278,0.643
-> other word,0.722,0.357


## 2. Errors that every seed makes

A clip misclassified by all seeds is a property of the clip (or of the training data), not of one model's initialisation.

In [4]:
clips = pd.read_csv(REPO / "data_accented" / "clips.csv")
acc = pd.DataFrame({
    "path": runs[0]["accented_path"],
    "speaker": runs[0]["accented_speaker"],
    "true": [LABELS[t] for t in runs[0]["accented_y_true"]],
})
preds = np.stack([r["accented_y_pred"] for r in runs])
acc["n_wrong"] = (preds != runs[0]["accented_y_true"]).sum(axis=0)
acc["preds"] = [", ".join(LABELS[p] for p in col) for col in preds.T]
acc = acc.merge(clips[["path", "word_s", "flag"]], on="path")

print(f"accented clips wrong for all {len(runs)} seeds: {(acc.n_wrong == len(runs)).sum()}, "
      f"for some: {(acc.n_wrong > 0).sum()}, of {len(acc)}")
acc[acc.n_wrong == len(runs)].sort_values(["true", "speaker"])[["path", "speaker", "true", "preds", "word_s", "flag"]]

accented clips wrong for all 3 seeds: 18, for some: 41, of 510


,path,speaker,true,preds,word_s,flag
230,clips/down/spk09_down_2.wav,spk09,down,"go, _unknown_, go",0.795,NaN
231,clips/down/spk09_down_3.wav,spk09,down,"go, go, go",0.865,NaN
460,clips/down/spk18_down_2.wav,spk18,down,"go, go, go",0.485,NaN
159,clips/go/spk06_go_1.wav,spk06,go,"off, off, off",0.405,NaN
233,clips/left/spk09_left_2.wav,spk09,left,"_unknown_, _unknown_, _unknown_",0.735,NaN
294,clips/no/spk12_no_2.wav,spk12,no,"_unknown_, _unknown_, _unknown_",0.575,NaN
295,clips/no/spk12_no_3.wav,spk12,no,"down, down, down",0.495,NaN
383,clips/no/spk15_no_1.wav,spk15,no,"down, _unknown_, _unknown_",0.565,NaN
503,clips/off/spk19_off_3.wav,spk19,off,"_unknown_, _unknown_, _unknown_",0.315,NaN
180,clips/on/spk07_on_3.wav,spk07,on,"_unknown_, _unknown_, right",0.425,NaN


In [5]:
off = pd.DataFrame({
    "path": runs[0]["official_path"],
    "true": [LABELS[t] for t in runs[0]["official_y_true"]],
})
# The unknown/silence sample in the test split depends on the seed; only the
# target-word clips are the same in every run.
target = runs[0]["official_is_target"]
off = off[target]
opreds = np.stack([r["official_y_pred"][target] for r in runs])
off["n_wrong"] = (opreds != runs[0]["official_y_true"][target]).sum(axis=0)
off["preds"] = [", ".join(LABELS[p] for p in col) for col in opreds.T]
always = off[off.n_wrong == len(runs)]
print(f"official target-word clips wrong for all seeds: {len(always)} of {len(off)}")
always.groupby("true").size().sort_values(ascending=False).rename("clips").to_frame().T

official target-word clips wrong for all seeds: 55 of 4074


true,on,down,up,off,go,right,no,left,stop,yes
clips,13,7,7,7,6,5,4,2,2,2


## 3. Per speaker and per word on the accented set

In [6]:
per_speaker = acc.groupby("speaker").agg(
    clips=("path", "size"),
    acc=("n_wrong", lambda w: 1 - w.mean() / len(runs)),
    words_wrong=("true", lambda t: ", ".join(sorted(set(t[acc.loc[t.index, "n_wrong"] > 0])))),
).sort_values("acc")
per_speaker["hand_cleaned"] = per_speaker.index.isin([f"spk{i:02d}" for i in range(7, 12)])
per_speaker.round(3).head(8)

,clips,acc,words_wrong,hand_cleaned
speaker,,,,
spk09,30,0.656,"down, left, no, off, on, right, up",True
spk16,10,0.833,"no, on, right",False
spk13,30,0.889,"on, right",False
spk06,10,0.900,go,False
spk12,30,0.922,"go, no",False
spk18,30,0.922,"down, on",False
spk15,30,0.944,"no, on, yes",False
spk19,30,0.956,"no, off",False


In [7]:
# Is the accent gap carried by a few speakers? Accuracy without the two weakest.
weakest = per_speaker.index[:2].tolist()
rest = acc[~acc.speaker.isin(weakest)]
print(f"all speakers: {1 - acc.n_wrong.mean() / len(runs):.4f}")
print(f"without {weakest}: {1 - rest.n_wrong.mean() / len(runs):.4f}")
print(f"official target words: {1 - off.n_wrong.mean() / len(runs):.4f}")

all speakers: 0.9451
without ['spk09', 'spk16']: 0.9660
official target words: 0.9643


## 4. Clip-level factors

Words longer than the 1 s window are cut (`flag = longer_than_clip`); speaking rate differs between the sets.

In [8]:
acc["duration_bin"] = pd.cut(acc.word_s, [0, 0.5, 0.7, 0.9, 1.0, 2.0])
acc["error_rate"] = acc.n_wrong / len(runs)
display(acc.groupby("duration_bin", observed=True).agg(clips=("path", "size"), error_rate=("error_rate", "mean")).round(3))
display(acc.groupby(acc.flag.fillna("")).agg(clips=("path", "size"), error_rate=("error_rate", "mean")).round(3))
print("median word duration by word:")
acc.groupby("true").word_s.median().sort_values().round(2).to_frame().T

,clips,error_rate
duration_bin,,
"(0.0, 0.5]",220,0.055
"(0.5, 0.7]",206,0.049
"(0.7, 0.9]",66,0.081
"(0.9, 1.0]",9,0.000
"(1.0, 2.0]",9,0.074


,clips,error_rate
flag,,
,501,0.055
longer_than_clip,9,0.074


median word duration by word:


true,up,off,on,go,left,right,no,stop,down,yes
word_s,0.36,0.42,0.46,0.52,0.52,0.56,0.56,0.57,0.58,0.62


## Notes

Fill in after listening to the clips in section 2 (use `listen(path)`, then clear the output):

- **Top failure mode:** _…_
- **Concrete example:** _…_
- **Hypothesis for why:** _…_